## Example 1 — TileLang and its TVM import with no GPU

In [ ]:
# The CUDA driver is opened on the first GPU call, not at import, so this cell runs without a GPU.
import tilelang
from tilelang import tvm
print(f'OK -- tilelang {tilelang.__version__} on TVM {tvm.__version__} from', tvm.__file__.split('/python/')[0])

## Example 2 — a kernel is turned into CUDA source for a named GPU family (no GPU needed)

In [ ]:
# Note: written for this notebook from TileLang's elementwise examples; the explicit target stands in for a GPU.
import tilelang
import tilelang.language as T
from tilelang import tvm

@T.prim_func
def add_one(A: T.Tensor((1024,), 'float32'), B: T.Tensor((1024,), 'float32')):
    with T.Kernel(8, threads=128) as bx:
        for i in T.Parallel(128):
            B[bx * 128 + i] = A[bx * 128 + i] + 1.0

target = tvm.target.Target({'kind': 'cuda', 'arch': 'sm_90'})
with target:
    artifact = tilelang.lower(add_one, target=target)
source = artifact.kernel_source
kernel_lines = [line for line in source.splitlines() if '__global__' in line]
assert any('__launch_bounds__(128' in line and 'add_one_kernel' in line for line in kernel_lines), kernel_lines
assert 'blockIdx.x' in source and 'threadIdx.x' in source, source
print('OK -- lowered to', len(source.splitlines()), 'lines of CUDA source:', kernel_lines[-1][:100])

## Example 3 — the swizzle-mode table (no GPU needed)

In [ ]:
from tilelang.layout import SwizzleMode
# name: (WGMMA layout type, TCGEN05 field, bytes, shared-memory alignment), from the class's own docstrings
expected = {'NONE': (0, 0, 1, 128), 'SWIZZLE_32B': (3, 6, 32, 256),
            'SWIZZLE_64B': (2, 4, 64, 512), 'SWIZZLE_128B': (1, 2, 128, 1024)}
for ordinal, (name, values) in enumerate(expected.items()):
    mode = getattr(SwizzleMode, name)
    got = (mode.wgmma_layout_type(), mode.tcgen05_layout_type(), mode.swizzle_byte_size(), mode.smem_alignment())
    assert got == values, (name, got, values)
    assert SwizzleMode.from_ordinal(ordinal).same_as(mode), (ordinal, name)
print('OK -- all four swizzle modes give their documented values, and each ordinal maps back to its mode')

## Example 4 — TileLang's quick-start kernel, compiled and checked against torch (GPU)

In [ ]:
import torch, tilelang, tilelang.language as T
# Adapted from TileLang's README quick-start (v0.1.15, MIT; the same code as v0.1.14's): a tiled half-precision
# matrix multiply with a ReLU at the end. Modified: condensed into one cell, and it prints which run-time compiler built the kernel.
@tilelang.jit
def matmul_relu(A, B, block_M: int = 128, block_N: int = 128, block_K: int = 32):
    M, N, K = T.const('M, N, K')
    A: T.Tensor((M, K), T.float16)
    B: T.Tensor((K, N), T.float16)
    C = T.empty((M, N), T.float16)
    with T.Kernel(T.ceildiv(N, block_N), T.ceildiv(M, block_M), threads=128) as (bx, by):
        A_shared = T.alloc_shared((block_M, block_K), T.float16)
        B_shared = T.alloc_shared((block_K, block_N), T.float16)
        C_local = T.alloc_fragment((block_M, block_N), T.float32)
        T.clear(C_local)
        for k in T.Pipelined(T.ceildiv(K, block_K), num_stages=3):
            T.copy(A[by * block_M, k * block_K], A_shared)
            T.copy(B[k * block_K, bx * block_N], B_shared)
            T.gemm(A_shared, B_shared, C_local)
        for i, j in T.Parallel(block_M, block_N):
            C_local[i, j] = T.max(C_local[i, j], 0)
        T.copy(C_local, C[by * block_M, bx * block_N])
    return C
a, b = (torch.randn((1024, 1024), device='cuda', dtype=torch.float16) for _ in range(2))
adapter = type(matmul_relu.compile(a, b).adapter).__name__  # which run-time compiler built (or cached) the kernel
torch.testing.assert_close(matmul_relu(a, b), torch.relu(a @ b), rtol=1e-2, atol=1e-2)
print('OK -- the kernel was compiled by', adapter, 'and matches torch.relu(a @ b) on', torch.cuda.get_device_name(0))